[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/07_%E3%83%9B%E3%83%93%E3%83%BC_%E9%99%B6%E8%8A%B83D/04_%E5%AE%9F%E7%94%A8%E3%83%84%E3%83%BC%E3%83%AB%E3%82%92%E3%81%A4%E3%81%8F%E3%82%8B.ipynb)

# ホビー-04. 実用ツールをつくる（スタンプ・こて・抜き型）

> 🟢 Colabで実行できます。**最初に「① セットアップ」**を実行。

> 🏺 **クレイプリンタが無くてもOK**。ふつうの3Dプリンタ（PLA）で印刷して**すぐ使える道具**を作ります。陶芸で一番手軽な3D活用です。

In [ ]:
# === ① セットアップ（最初に実行）===
try:
    from solid2 import cylinder, cube, circle, linear_extrude, translate
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','solidpython2'])
    from solid2 import cylinder, cube, circle, linear_extrude, translate
import math
print('準備OK')

## 1. スタンプ（型押し）

底面のドット模様を粘土に押しつけて模様をつけます。上に持ち手付き。

In [ ]:
def make_stamp(diameter=40, thickness=8, dot_d=4, dot_h=3, n=8, fn=80):
    disk = cylinder(d=diameter, h=thickness, _fn=fn)                    # 本体の円盤
    handle = translate([0,0,thickness])(cylinder(d=diameter*0.4, h=25, _fn=40))  # 持ち手
    dots = translate([0,0,-dot_h])(cylinder(d=dot_d, h=dot_h, _fn=24))  # 中心のドット（下面）
    r = diameter*0.3
    for i in range(n):                                                  # 周囲にドットを並べる
        a = 2*math.pi*i/n
        dots += translate([r*math.cos(a), r*math.sin(a), -dot_h])(cylinder(d=dot_d, h=dot_h, _fn=24))
    return disk + handle + dots

make_stamp(n=8).save_as_scad('stamp.scad')
print('stamp.scad を書き出しました')

> 💡 ドット模様は左右対称なので問題ありませんが、**文字を入れると粘土側では鏡像**になります（入れるなら反転を）。

## 2. プロファイルこて（リブ）

板の片側を曲線にえぐった道具。ろくろで器の内/外を**一定の曲面にならす**のに使います。

In [ ]:
def make_rib(radius=60, plate_w=80, plate_h=60, t=4):
    plate = cube([plate_w, plate_h, t])                                  # 平らな板
    arc = translate([plate_w+radius-15, plate_h/2, -1])(cylinder(r=radius, h=t+2, _fn=120))  # 円で削る
    return plate - arc                                                   # 板から円弧を引く＝凹カーブ

make_rib(radius=60).save_as_scad('rib.scad')
print('rib.scad を書き出しました（radiusを変えるとカーブが変わる）')

## 3. 抜き型（クッキーカッター式）

薄い壁の輪っか。たたら（板作り）で粘土を**同じ形に抜く**のに使います。`fn` で形が変わります（円=100, 六角=6）。

In [ ]:
def make_cutter(size=70, wall=1.5, height=25, fn=100):
    outer = circle(d=size, _fn=fn)                 # 外側の輪郭
    inner = circle(d=size-2*wall, _fn=fn)          # 内側（壁の厚みぶん小さく）
    return linear_extrude(height)(outer - inner)   # 輪郭の差を上に押し出す＝壁

make_cutter(size=70, fn=100).save_as_scad('cutter_circle.scad')   # 丸
make_cutter(size=70, fn=6).save_as_scad('cutter_hex.scad')        # 六角
print('cutter_circle.scad / cutter_hex.scad を書き出しました')

In [ ]:
import glob, os
for f in sorted(glob.glob('*.scad')):
    print(f, f'({os.path.getsize(f)} bytes)')

---
## 🏆 練習問題

**問1.** `make_stamp` の `n`（ドットの数）を 12 にして印刷データを作ろう。

**問2.** `make_cutter` で **五角形**（fn=5）の抜き型を作ろう。

**問3.** `make_rib` の `radius` を 40 と 100 で作り、カーブの違いを OpenSCAD で見比べよう。

In [ ]:
# 問1〜3


> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/07_%E3%83%9B%E3%83%93%E3%83%BC_%E9%99%B6%E8%8A%B83D/04_%E5%AE%9F%E7%94%A8%E3%83%84%E3%83%BC%E3%83%AB%E3%82%92%E3%81%A4%E3%81%8F%E3%82%8B.md)**

## 📒 用語集 ＆ チートシート

| 道具 | 何に使う |
|---|---|
| スタンプ | 模様・ロゴを型押し |
| こて(リブ) | ろくろで曲面をならす |
| 抜き型 | 板を同じ形に抜く |
| sprig | 別作りの飾りを貼る型 |

印刷したら：OpenSCADで `.scad`→`.stl` 書き出し → スライサー(Cura等)で印刷。

In [ ]:
# チートシート（SolidPythonの基本操作）
cylinder(d=40, h=8, _fn=80)      # 円柱
cube([80, 60, 4])               # 直方体
circle(d=70, _fn=6)             # 2D 六角
linear_extrude(25)(circle(d=70))  # 2D→3D 押し出し
translate([10,0,0])(cube([5,5,5]))   # 移動
# A + B = 合体,  A - B = くり抜き